# S6E10 Airline Passenger Satisfaction — Modelling & Training

**Metric:** ROC-AUC  
**Target:** `satisfaction` (1 = satisfied, 0 = not satisfied)  
**Feature set:** 23 columns from FE round 1 + `te_flight_distance` computed inside each fold (FE round 2)  
**Validation:** 5-fold StratifiedKFold (shuffle=True, random_state=42)  

---

## Contents
1. [Imports & Setup](#1-imports--setup)
2. [Load Data](#2-load-data)
3. [Target Encoding Helper](#3-target-encoding-helper)
4. [LightGBM Baseline](#4-lightgbm-baseline)
5. [CatBoost Baseline](#5-catboost-baseline)
6. [XGBoost Baseline](#6-xgboost-baseline)
7. [Baseline Ensemble & Correlation](#7-baseline-ensemble--correlation)
8. [LightGBM — Optuna Tuning](#8-lightgbm--optuna-tuning)
9. [LightGBM — Tuned (5-fold)](#9-lightgbm--tuned-5-fold)
10. [CatBoost — Optuna Tuning](#10-catboost--optuna-tuning)
11. [CatBoost — Tuned (5-fold)](#11-catboost--tuned-5-fold)
12. [Final Ensemble](#12-final-ensemble)
13. [Results Summary](#13-results-summary)
14. [Submissions](#14-submissions)

## 1. Imports & Setup

In [ ]:
import os, numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder
from catboost import CatBoostClassifier
import lightgbm as lgb
import xgboost as xgb
import optuna
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Works both locally and on Kaggle
DATA_DIR = Path('/kaggle/input/competitions/playground-series-s6e10') if Path('/kaggle/input/competitions/playground-series-s6e10').exists() \
           else Path('data/raw')

N_FOLDS = 5
RANDOM_STATE = 42
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

print(f"Data directory: {DATA_DIR}")

## 2. Load Data & Preprocessing

Reads raw `train.csv` / `test.csv`, applies the same preprocessing as `features.ipynb` (missing-value flag, fillna, target encoding), then adds the two frequency features from FE round 1 (`flight_distance_freq`, `flight_distance_type_of_travel_freq`). The result is 23 features ready for modelling. The in-fold target encoding (`te_flight_distance`) is added inside each CV loop in sections 4–11.

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')

# Normalise column names: 'Inflight wifi service' -> 'inflight_wifi_service'
train.columns = train.columns.str.lower().str.replace(' ', '_')
test.columns  = test.columns.str.lower().str.replace(' ', '_')

# --- Preprocessing (identical to features.ipynb) ---

# 1. Missing-value flag — before fillna, while the gap information still exists
train['arrival_delay_missing'] = train['arrival_delay_in_minutes'].isna().astype(int)
test['arrival_delay_missing']  = test['arrival_delay_in_minutes'].isna().astype(int)

# 2. Fill gaps in arrival_delay with departure_delay (correlation 0.97)
train['arrival_delay_in_minutes'] = train['arrival_delay_in_minutes'].fillna(train['departure_delay_in_minutes'])
test['arrival_delay_in_minutes']  = test['arrival_delay_in_minutes'].fillna(test['departure_delay_in_minutes'])

# 3. Target: True/False -> 1/0
y_train = train['satisfaction'].astype(int)

# 4. Drop id, target, and arrival_delay (near-duplicate of departure_delay)
CAT_COLS  = ['gender', 'customer_type', 'type_of_travel', 'class']
DROP_COLS = ['id', 'satisfaction', 'arrival_delay_in_minutes']

X_raw      = train.drop(columns=DROP_COLS)
X_test_raw = test.drop(columns=['id', 'arrival_delay_in_minutes'])

# --- Add frequency features (E8 + E9 from features.ipynb round 1) ---

def add_features(train_df, test_df):
    """Add flight_distance_freq and flight_distance_type_of_travel_freq.
    Counts are computed over train+test together (no target involved)."""
    train_df = train_df.copy()
    test_df  = test_df.copy()
    full = pd.concat([train_df, test_df], ignore_index=True)
    n = len(train_df)

    # E8: frequency of the flight_distance value
    counts = full['flight_distance'].map(full['flight_distance'].value_counts())
    train_df['flight_distance_freq'] = counts.iloc[:n].values
    test_df['flight_distance_freq']  = counts.iloc[n:].values

    # E9: frequency of (flight_distance, type_of_travel) pair
    key = full['flight_distance'].astype(str) + '_' + full['type_of_travel'].astype(str)
    counts = key.map(key.value_counts())
    train_df['flight_distance_type_of_travel_freq'] = counts.iloc[:n].values
    test_df['flight_distance_type_of_travel_freq']  = counts.iloc[n:].values

    return train_df, test_df

X_train, X_test = add_features(X_raw, X_test_raw)

print(f'Train: {X_train.shape}, Test: {X_test.shape}, Target: {y_train.shape}')
print(f'Positive rate: {y_train.mean():.3f}')
print(f'Features: {list(X_train.columns)}')

## 3. Target Encoding Helper

**Why inside the fold?**  
`te_flight_distance` encodes the target rate per distance value. If computed on the full train before splitting, the validation fold leaks its own target into the feature — CV score inflates and becomes unreliable. Computing it inside each fold (fit on train part only, transform val and test) avoids this leak.

**`cv=5` inside TargetEncoder:** the encoder itself uses cross-fitting on the training part so that even training rows get an out-of-fold estimate. This prevents the model from memorising the encoding.

In [ ]:
def add_te_flight_distance(X_fit, y_fit, other_frames, smooth='auto'):
    """Add te_flight_distance column without target leakage.
    
    Call INSIDE a fold:
      X_fit / y_fit  — training part of the fold
      other_frames   — list of frames to transform with the fitted encoder
                       (validation part, test set)
    Returns: encoded X_fit, list of encoded other frames.
    """
    enc = TargetEncoder(smooth=smooth, cv=5, shuffle=True, random_state=RANDOM_STATE)
    key = lambda d: d[['flight_distance']].astype(str)

    X_fit = X_fit.copy()
    X_fit['te_flight_distance'] = enc.fit_transform(key(X_fit), y_fit)[:, 0]

    encoded = []
    for frame in other_frames:
        frame = frame.copy()
        frame['te_flight_distance'] = enc.transform(key(frame))[:, 0]
        encoded.append(frame)

    return X_fit, encoded

## 4. LightGBM Baseline

Baseline parameters from `features.ipynb`. Used as the reference point throughout FE (CV 0.95964 on 23 features without TE).

In [ ]:
# Cast categoricals for LightGBM
X_train_lgbm = X_train.copy()
X_test_lgbm  = X_test.copy()
for col in CAT_COLS:
    X_train_lgbm[col] = X_train_lgbm[col].astype('category')
    X_test_lgbm[col]  = X_test_lgbm[col].astype('category')

LGBM_PARAMS = {
    'objective': 'binary', 'metric': 'auc',
    'learning_rate': 0.05, 'num_leaves': 127, 'min_child_samples': 20,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'n_estimators': 1000, 'early_stopping_rounds': 50,
    'verbose': -1, 'random_state': RANDOM_STATE,
    'device': 'gpu',
}

oof_lgbm  = np.zeros(len(X_train))
test_lgbm = np.zeros(len(X_test))
fold_scores_lgbm = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_lgbm, y_train), 1):
    X_tr, X_val = X_train_lgbm.iloc[tr_idx], X_train_lgbm.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    X_tr, (X_val, X_te) = add_te_flight_distance(X_tr, y_tr, [X_val, X_test_lgbm])

    model = lgb.LGBMClassifier(**LGBM_PARAMS)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], categorical_feature=CAT_COLS)

    val_pred = model.predict_proba(X_val)[:, 1]
    oof_lgbm[val_idx] = val_pred
    test_lgbm += model.predict_proba(X_te)[:, 1] / N_FOLDS

    score = roc_auc_score(y_val, val_pred)
    fold_scores_lgbm.append(score)
    print(f'Fold {fold} AUC: {score:.5f}')

cv_lgbm = roc_auc_score(y_train, oof_lgbm)
print(f'\nCV AUC (OOF): {cv_lgbm:.5f}')
print(f'Mean fold:    {np.mean(fold_scores_lgbm):.5f} ± {np.std(fold_scores_lgbm):.5f}')

## 5. CatBoost Baseline

CatBoost uses **oblivious (symmetric) trees**: all branches at the same depth make the same split. This differs from LightGBM's leaf-wise growth. `depth=6` (64 leaves) is the standard working range for CatBoost — comparable in regularisation strength to `num_leaves=127` in LightGBM.

CatBoost accepts string categoricals natively and builds its own internal target statistics (CTR/Counter) per category and combination. No manual casting to `category` dtype needed.

In [ ]:
CAT_PARAMS = {
    'iterations': 1000, 'learning_rate': 0.05, 'depth': 6,
    'loss_function': 'Logloss', 'eval_metric': 'AUC',
    'early_stopping_rounds': 50, 'random_seed': RANDOM_STATE,
    'thread_count': -1, 'verbose': 0,
    'task_type': 'GPU',
}

oof_cat  = np.zeros(len(X_train))
test_cat = np.zeros(len(X_test))
fold_scores_cat = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    X_tr, (X_val, X_te) = add_te_flight_distance(X_tr, y_tr, [X_val, X_test])

    model = CatBoostClassifier(**CAT_PARAMS, cat_features=CAT_COLS)
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    val_pred = model.predict_proba(X_val)[:, 1]
    oof_cat[val_idx] = val_pred
    test_cat += model.predict_proba(X_te)[:, 1] / N_FOLDS

    score = roc_auc_score(y_val, val_pred)
    fold_scores_cat.append(score)
    print(f'Fold {fold} AUC: {score:.5f}')

cv_cat = roc_auc_score(y_train, oof_cat)
print(f'\nCV AUC (OOF): {cv_cat:.5f}')
print(f'Mean fold:    {np.mean(fold_scores_cat):.5f} ± {np.std(fold_scores_cat):.5f}')

## 6. XGBoost Baseline

`tree_method='hist'` enables histogram-based splitting — the same algorithm LightGBM uses by default. Without it, XGBoost would be ~10× slower on this dataset size.

`enable_categorical=True` requires columns to have pandas `category` dtype.

In [ ]:
# Cast categoricals for XGBoost
X_train_xgb = X_train.copy()
X_test_xgb  = X_test.copy()
for col in CAT_COLS:
    X_train_xgb[col] = X_train_xgb[col].astype('category')
    X_test_xgb[col]  = X_test_xgb[col].astype('category')

XGB_PARAMS = {
    'n_estimators': 1000, 'learning_rate': 0.05, 'max_depth': 6,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'tree_method': 'hist', 'enable_categorical': True,
    'device': 'cuda',
    'eval_metric': 'auc', 'early_stopping_rounds': 50,
    'random_state': RANDOM_STATE,
}

oof_xgb  = np.zeros(len(X_train))
test_xgb = np.zeros(len(X_test))
fold_scores_xgb = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_xgb, y_train), 1):
    X_tr, X_val = X_train_xgb.iloc[tr_idx], X_train_xgb.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    X_tr, (X_val, X_te) = add_te_flight_distance(X_tr, y_tr, [X_val, X_test_xgb])

    model = xgb.XGBClassifier(**XGB_PARAMS)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=0)

    val_pred = model.predict_proba(X_val)[:, 1]
    oof_xgb[val_idx] = val_pred
    test_xgb += model.predict_proba(X_te)[:, 1] / N_FOLDS

    score = roc_auc_score(y_val, val_pred)
    fold_scores_xgb.append(score)
    print(f'Fold {fold} AUC: {score:.5f}')

cv_xgb = roc_auc_score(y_train, oof_xgb)
print(f'\nCV AUC (OOF): {cv_xgb:.5f}')
print(f'Mean fold:    {np.mean(fold_scores_xgb):.5f} ± {np.std(fold_scores_xgb):.5f}')

## 7. Baseline Ensemble & Correlation

Before tuning, we check whether the three models are diverse enough to benefit from ensembling.

**OOF correlation** tells us how similarly models rank the same rows. Correlation > 0.99 means models make nearly identical errors — averaging them gives little gain over the best single model.

In [ ]:
oof_df = pd.DataFrame({'lgbm': oof_lgbm, 'xgb': oof_xgb, 'cat': oof_cat})
print('OOF prediction correlation:')
print(oof_df.corr().round(4))

oof_ensemble_base  = (oof_lgbm + oof_xgb + oof_cat) / 3
test_ensemble_base = (test_lgbm + test_xgb + test_cat) / 3

cv_ensemble_base = roc_auc_score(y_train, oof_ensemble_base)
print(f'\nBaseline ensemble CV AUC: {cv_ensemble_base:.5f}')

print(f'\n--- Baseline summary ---')
print(f'LightGBM : {cv_lgbm:.5f}')
print(f'XGBoost  : {cv_xgb:.5f}')
print(f'CatBoost : {cv_cat:.5f}')
print(f'Ensemble : {cv_ensemble_base:.5f}')

**Conclusion:** OOF correlations are 0.997–0.998 — all three models find the same patterns. Simple averaging does not improve over the best single model. Hyperparameter tuning is the next lever.

## 8. LightGBM — Optuna Tuning

**Search space:**
- `learning_rate` — log scale (equal probability for 0.01–0.02 and 0.05–0.1)
- `num_leaves` — controls tree complexity (LightGBM grows leaf-wise)
- `reg_alpha / reg_lambda` — L1 and L2 regularisation, not used in the baseline

**3-fold CV** is used inside Optuna (instead of 5-fold) to keep runtime at ~90 min for 50 trials.
Final evaluation of the best parameters is done on 5-fold in section 9.

In [ ]:
def objective_lgbm(trial):
    params = {
        'objective': 'binary', 'metric': 'auc',
        'verbose': -1, 'random_state': RANDOM_STATE,
        'n_estimators': 1000, 'early_stopping_rounds': 50,
        'device': 'gpu',
        'learning_rate':      trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves':         trial.suggest_int('num_leaves', 63, 255),
        'min_child_samples':  trial.suggest_int('min_child_samples', 10, 100),
        'subsample':          trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':   trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha':          trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':         trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
    }
    skf_opt = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    for tr_idx, val_idx in skf_opt.split(X_train_lgbm, y_train):
        X_tr, X_val = X_train_lgbm.iloc[tr_idx], X_train_lgbm.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        X_tr, (X_val, _) = add_te_flight_distance(X_tr, y_tr, [X_val, X_test_lgbm])
        model = lgb.LGBMClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], categorical_feature=CAT_COLS)
        scores.append(roc_auc_score(y_val, model.predict_proba(X_val)[:, 1]))
    return np.mean(scores)

study_lgbm = optuna.create_study(direction='maximize')
study_lgbm.optimize(objective_lgbm, n_trials=50, show_progress_bar=True)

print(f'Best CV (3-fold): {study_lgbm.best_value:.5f}')
print(f'Best params: {study_lgbm.best_params}')

## 9. LightGBM — Tuned (5-fold)

Re-train with the best parameters found by Optuna on the full 5-fold scheme for a fair comparison with baselines.

In [ ]:
LGBM_OPT_PARAMS = {
    'objective': 'binary', 'metric': 'auc',
    'verbose': -1, 'random_state': RANDOM_STATE,
    'n_estimators': 1000, 'early_stopping_rounds': 50,
    **study_lgbm.best_params,
}

oof_lgbm_opt  = np.zeros(len(X_train))
test_lgbm_opt = np.zeros(len(X_test))
fold_scores_lgbm_opt = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_lgbm, y_train), 1):
    X_tr, X_val = X_train_lgbm.iloc[tr_idx], X_train_lgbm.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    X_tr, (X_val, X_te) = add_te_flight_distance(X_tr, y_tr, [X_val, X_test_lgbm])

    model = lgb.LGBMClassifier(**LGBM_OPT_PARAMS)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], categorical_feature=CAT_COLS)

    val_pred = model.predict_proba(X_val)[:, 1]
    oof_lgbm_opt[val_idx] = val_pred
    test_lgbm_opt += model.predict_proba(X_te)[:, 1] / N_FOLDS

    score = roc_auc_score(y_val, val_pred)
    fold_scores_lgbm_opt.append(score)
    print(f'Fold {fold} AUC: {score:.5f}')

cv_lgbm_opt = roc_auc_score(y_train, oof_lgbm_opt)
print(f'\nCV AUC (OOF): {cv_lgbm_opt:.5f}')
print(f'Mean fold:    {np.mean(fold_scores_lgbm_opt):.5f} ± {np.std(fold_scores_lgbm_opt):.5f}')
print(f'vs baseline:  {cv_lgbm_opt - cv_lgbm:+.5f}')

## 10. CatBoost — Optuna Tuning

CatBoost-specific parameters:
- `depth` — tree depth (capped at 6 for speed; oblivious trees grow exponentially slower with depth)
- `l2_leaf_reg` — L2 regularisation on leaf weights
- `bagging_temperature` — controls randomness of Bayesian bootstrap (0 = no randomness)
- `random_strength` — adds noise to split scores to reduce overfitting

**30% subsample** of train used for speed (full dataset would take ~25 hours for 50 trials).

In [ ]:
X_opt, _, y_opt, _ = train_test_split(
    X_train, y_train, train_size=0.3, stratify=y_train, random_state=RANDOM_STATE
)

def objective_cat(trial):
    params = {
        'iterations': 1000, 'loss_function': 'Logloss', 'eval_metric': 'AUC',
        'early_stopping_rounds': 50, 'random_seed': RANDOM_STATE,
        'verbose': 0, 'thread_count': -1,
        'task_type': 'GPU',
        'learning_rate':      trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth':              trial.suggest_int('depth', 4, 6),
        'l2_leaf_reg':        trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True),
        'bagging_temperature':trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength':    trial.suggest_float('random_strength', 0.0, 1.0),
        'min_data_in_leaf':   trial.suggest_int('min_data_in_leaf', 1, 50),
    }
    skf_opt = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    for tr_idx, val_idx in skf_opt.split(X_opt, y_opt):
        X_tr, X_val = X_opt.iloc[tr_idx], X_opt.iloc[val_idx]
        y_tr, y_val = y_opt.iloc[tr_idx], y_opt.iloc[val_idx]
        X_tr, (X_val, _) = add_te_flight_distance(X_tr, y_tr, [X_val, X_test])
        model = CatBoostClassifier(**params, cat_features=CAT_COLS)
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val))
        scores.append(roc_auc_score(y_val, model.predict_proba(X_val)[:, 1]))
    return np.mean(scores)

study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(objective_cat, n_trials=50, show_progress_bar=True)

print(f'Best CV (3-fold, 30% data): {study_cat.best_value:.5f}')
print(f'Best params: {study_cat.best_params}')

## 11. CatBoost — Tuned (5-fold)

In [ ]:
CAT_OPT_PARAMS = {
    'iterations': 1000, 'loss_function': 'Logloss', 'eval_metric': 'AUC',
    'early_stopping_rounds': 50, 'random_seed': RANDOM_STATE,
    'thread_count': -1, 'verbose': 0,
    **study_cat.best_params,
}

oof_cat_opt  = np.zeros(len(X_train))
test_cat_opt = np.zeros(len(X_test))
fold_scores_cat_opt = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    X_tr, (X_val, X_te) = add_te_flight_distance(X_tr, y_tr, [X_val, X_test])

    model = CatBoostClassifier(**CAT_OPT_PARAMS, cat_features=CAT_COLS)
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    val_pred = model.predict_proba(X_val)[:, 1]
    oof_cat_opt[val_idx] = val_pred
    test_cat_opt += model.predict_proba(X_te)[:, 1] / N_FOLDS

    score = roc_auc_score(y_val, val_pred)
    fold_scores_cat_opt.append(score)
    print(f'Fold {fold} AUC: {score:.5f}')

cv_cat_opt = roc_auc_score(y_train, oof_cat_opt)
print(f'\nCV AUC (OOF): {cv_cat_opt:.5f}')
print(f'Mean fold:    {np.mean(fold_scores_cat_opt):.5f} ± {np.std(fold_scores_cat_opt):.5f}')
print(f'vs baseline:  {cv_cat_opt - cv_cat:+.5f}')

## 12. Final Ensemble

We check OOF correlation between the two tuned models and test all ensemble combinations.

In [ ]:
oof_df_final = pd.DataFrame({
    'lgbm_opt': oof_lgbm_opt,
    'cat_opt':  oof_cat_opt,
    'xgb':      oof_xgb,
})
print('OOF correlation (tuned models):')
print(oof_df_final.corr().round(4))

# Simple averages
combos = {
    'lgbm_opt + cat_opt':       (oof_lgbm_opt + oof_cat_opt) / 2,
    'lgbm_opt + xgb':           (oof_lgbm_opt + oof_xgb) / 2,
    'lgbm_opt + cat_opt + xgb': (oof_lgbm_opt + oof_cat_opt + oof_xgb) / 3,
}
print('\nEnsemble CV AUC:')
for name, oof in combos.items():
    print(f'  {name}: {roc_auc_score(y_train, oof):.5f}')

# Best ensemble for submission
oof_final  = (oof_lgbm_opt + oof_cat_opt) / 2
test_final = (test_lgbm_opt + test_cat_opt) / 2
cv_final = roc_auc_score(y_train, oof_final)
print(f'\nSelected ensemble CV AUC: {cv_final:.5f}')

## 13. Results Summary

In [ ]:
results = pd.DataFrame([
    {'Model': 'LightGBM baseline',    'CV AUC': cv_lgbm},
    {'Model': 'CatBoost baseline',    'CV AUC': cv_cat},
    {'Model': 'XGBoost baseline',     'CV AUC': cv_xgb},
    {'Model': 'Baseline ensemble',    'CV AUC': cv_ensemble_base},
    {'Model': 'LightGBM + Optuna',    'CV AUC': cv_lgbm_opt},
    {'Model': 'CatBoost + Optuna',    'CV AUC': cv_cat_opt},
    {'Model': 'Final ensemble',       'CV AUC': cv_final},
])
results['Delta vs LGBM baseline'] = results['CV AUC'] - cv_lgbm
results = results.sort_values('CV AUC', ascending=False).reset_index(drop=True)
print(results.to_string(index=False, float_format='{:.5f}'.format))

## 14. Submissions

In [ ]:
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')

# LightGBM tuned
sub_lgbm = sample_sub.copy()
sub_lgbm['satisfaction'] = test_lgbm_opt
sub_lgbm.to_csv('submission_lgbm_optuna_te.csv', index=False)
print(f'submission_lgbm_optuna_te.csv — CV {cv_lgbm_opt:.5f}')

# CatBoost tuned
sub_cat = sample_sub.copy()
sub_cat['satisfaction'] = test_cat_opt
sub_cat.to_csv('submission_cat_optuna_te.csv', index=False)
print(f'submission_cat_optuna_te.csv  — CV {cv_cat_opt:.5f}')

# Final ensemble
sub_final = sample_sub.copy()
sub_final['satisfaction'] = test_final
sub_final.to_csv('submission_ensemble_final.csv', index=False)
print(f'submission_ensemble_final.csv — CV {cv_final:.5f}')